# 02 — Retrieval with six broad subject groups

This experiment asks whether retrieval comparisons change when the 20 Newsgroups categories are evaluated as six broader subject groups. Related subtopics count as relevant to the same broad query. The grouping is a human-defined evaluation choice; PLSA still learns from word counts without category labels.

We compare PLSA, TF-IDF/VSM, and LSI on the same held-out documents, inspect how PLSA topics align with the six groups, and finish with a PLSA-only controlled-mixture probe. The shared corpus comes from `src/preprocessing.ipynb`; this notebook defines its own grouping, split, and fitted representations.

Run from the `Local` repository root. Results replace the previous files in `results/six_category_experiment/`.

## 1. Settings and imports

We keep all documents, use a stratified 80/20 split, and fit six PLSA topics. Matching the number of topics to the number of groups is a modelling choice for this run; it does not force a one-to-one topic/group mapping.

In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

REPOSITORY_ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation import evaluate_ranked_retrieval
from src.lsi import fit_lsi
from src.plsa_model import train_plsa_restarts
from src.retrieval import cosine_scores
from src.synthetic_data import make_mixed_document
from src.text_processing import load_nlp, preprocess_texts
from src.tfidf import fit_tfidf

SEED = 42
TEST_SIZE = 0.20
K = 6
N_RESTARTS = 5
MAX_ITER = 100
TOLERANCE = 1e-4
MAX_FEATURES = 3000
MIN_DF = 3
MAX_DF = 0.90
RETRIEVAL_K = 10
DATA_PATH = REPOSITORY_ROOT / 'data' / '20news_preprocessed.npz'
OUTPUT_DIR = REPOSITORY_ROOT / 'results' / 'six_category_experiment'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPOSITORY_ROOT.resolve()}')
print(f'Results folder: {OUTPUT_DIR.resolve()}')

## 2. Load the corpus and define the six groups

Each original category belongs to exactly one broad group. We retain original labels for group diagnostics and mixture sampling. The six-group labels define stratification and retrieval relevance; neither label set is used to train PLSA.

In [ ]:
SUPERGROUPS = {
    'Computing': ['comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x'],
    'Digital technology and engineering': ['sci.crypt', 'sci.electronics'],
    'Health and space science': ['sci.med', 'sci.space'],
    'Recreation and consumer exchange': ['misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey'],
    'Religion and worldview': ['alt.atheism', 'soc.religion.christian', 'talk.religion.misc'],
    'Politics and international affairs': ['talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc'],
}

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run src/preprocessing.ipynb first.')
with np.load(DATA_PATH, allow_pickle=False) as corpus:
    documents = corpus['documents'].tolist()
    original_labels = corpus['labels'].astype(np.int64, copy=False)
    source_category_names = corpus['category_names'].tolist()

mapped_categories = [category for members in SUPERGROUPS.values() for category in members]
missing = sorted(set(source_category_names) - set(mapped_categories))
unknown = sorted(set(mapped_categories) - set(source_category_names))
duplicates = sorted({name for name in mapped_categories if mapped_categories.count(name) > 1})
if missing or unknown or duplicates:
    raise ValueError(f'Each source category must appear once. Missing={missing}; unknown={unknown}; duplicates={duplicates}')
source_to_supergroup = {category: group for group, members in SUPERGROUPS.items() for category in members}
target_names = list(SUPERGROUPS)
supergroup_to_id = {name: i for i, name in enumerate(target_names)}
group_labels = np.asarray([supergroup_to_id[source_to_supergroup[source_category_names[label]]] for label in original_labels], dtype=np.int64)

category_counts = pd.DataFrame([
    {'supergroup': group, 'source_category': category,
     'documents': int(np.count_nonzero(original_labels == source_category_names.index(category)))}
    for group, members in SUPERGROUPS.items() for category in members
])
group_counts = pd.DataFrame({'supergroup': target_names, 'documents': np.bincount(group_labels, minlength=len(target_names))})
print(f'Documents: {len(documents):,}; source categories: {len(source_category_names)}')
display(category_counts)
display(group_counts)

## 3. Split the data and build count features

We stratify on the six groups and fit the count vocabulary on training text only. Test documents use that same vocabulary. Documents that become empty after preprocessing are excluded from retrieval candidates so that all methods rank the same usable set.

In [ ]:
docs_train, docs_test, y_train, y_test, original_y_train, original_y_test = train_test_split(
    documents, group_labels, original_labels, test_size=TEST_SIZE,
    random_state=SEED, stratify=group_labels,
)
nonempty_test_mask = np.asarray([bool(text.strip()) for text in docs_test])
print('Training documents by group:', dict(zip(target_names, np.bincount(y_train, minlength=len(target_names)))))
print(f'Train/test documents: {len(docs_train):,}/{len(docs_test):,}')
print(f'Non-empty test candidates: {int(nonempty_test_mask.sum()):,}/{len(docs_test):,}')

from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer(
    lowercase=False, tokenizer=str.split, token_pattern=None,
    max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF, dtype=np.float64,
)
X_train = count_vectorizer.fit_transform(docs_train)
X_test = count_vectorizer.transform(docs_test)
vocabulary = count_vectorizer.get_feature_names_out()
print('Count matrices (train/test):', X_train.shape, X_test.shape)

## 4. Fit PLSA, TF-IDF, and LSI

PLSA uses five random initializations and selects the run with the highest training log-likelihood. TF-IDF and LSI are fitted using training documents only. All three methods then represent the held-out test set.

In [ ]:
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=N_RESTARTS,
    max_iter=MAX_ITER, tol=TOLERANCE, base_seed=SEED,
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
restart_summary = pd.DataFrame([
    {'restart': i + 1, 'seed': SEED + i, 'iterations': model.n_iter_,
     'training_log_likelihood': float(ll), 'selected': model is best_plsa}
    for i, (model, ll) in enumerate(restart_runs)
])
display(restart_summary)

tfidf_vectorizer, (X_train_tfidf, X_test_tfidf) = fit_tfidf(
    docs_train, docs_test, max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
)
lsi_model, (lsi_train, lsi_test) = fit_lsi(
    X_train_tfidf, X_test_tfidf, n_components=50, seed=SEED,
)
print('PLSA topic mixtures:', theta_train.shape, theta_test.shape)
print('TF-IDF matrices:', X_train_tfidf.shape, X_test_tfidf.shape)
print('LSI matrices:', lsi_train.shape, lsi_test.shape)

## 5. Inspect PLSA topics and group alignment

The top words and convergence curve describe the selected PLSA fit. NMI and ARI compare each non-empty test document's dominant topic with its six-group label after unsupervised training. They summarize alignment; labels do not enter model fitting.

In [ ]:
topic_rows = []
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}: " + ', '.join(f'{word} ({prob:.3f})' for word, prob in words))
    topic_rows.extend(
        {'topic': topic_id, 'rank': rank, 'word': word, 'probability': probability}
        for rank, (word, probability) in enumerate(words, start=1)
    )
topic_words = pd.DataFrame(topic_rows)

convergence_fig, convergence_ax = plt.subplots(figsize=(8, 4))
convergence_ax.plot(range(1, len(best_plsa.log_likelihoods) + 1), best_plsa.log_likelihoods, marker='o', markersize=3)
convergence_ax.set(xlabel='EM iteration', ylabel='Training log-likelihood', title='PLSA convergence')
convergence_ax.grid(alpha=0.3)
convergence_fig.tight_layout()
display(convergence_fig)

test_topics = np.argmax(theta_test[nonempty_test_mask], axis=1)
test_groups = y_test[nonempty_test_mask]
topic_alignment = pd.DataFrame([{
    'NMI': normalized_mutual_info_score(test_groups, test_topics),
    'ARI': adjusted_rand_score(test_groups, test_topics),
    'documents': len(test_groups),
}])
display(topic_alignment.round(4))

## 6. Compare held-out retrieval performance

Each broad group has one fixed query. A test document is relevant when its original category belongs to the query's group. All methods rank the same non-empty test candidates. We report Precision@10, Recall@10, AP@10, and NDCG@10 per query, then average across the six queries to obtain MAP@10 and macro-average scores.

In [ ]:
supergroup_queries = {
    'Computing': 'computer graphics software hardware operating systems windows',
    'Digital technology and engineering': 'electronics circuits encryption cryptography hardware',
    'Health and space science': 'medicine health disease doctor space NASA planet',
    'Recreation and consumer exchange': 'cars motorcycles baseball hockey sports sale price',
    'Religion and worldview': 'religion atheist christian belief faith god',
    'Politics and international affairs': 'politics government guns international middle east policy',
}
if set(supergroup_queries) != set(target_names):
    raise ValueError('Provide exactly one query for each supergroup.')

query_nlp = load_nlp()
def evaluate_supergroup_query(query_text, target_label, target_name, k=RETRIEVAL_K):
    processed_query = preprocess_texts([query_text], nlp=query_nlp)[0]
    if not processed_query.strip():
        print(f'Warning: query for {target_name} became empty after preprocessing.')
    query_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    query_tfidf = tfidf_vectorizer.transform([processed_query])
    query_lsi = lsi_model.transform(query_tfidf)
    query_theta = best_plsa.fold_in(query_counts, max_iter=100)
    candidate_groups = y_test[nonempty_test_mask]
    method_scores = {
        'TF-IDF/VSM': cosine_scores(query_tfidf, X_test_tfidf[nonempty_test_mask]),
        'LSI': cosine_scores(query_lsi, lsi_test[nonempty_test_mask]),
        'PLSA': cosine_scores(query_theta, theta_test[nonempty_test_mask]),
    }
    rows = []
    for method, scores in method_scores.items():
        metric = evaluate_ranked_retrieval(scores, candidate_groups, target_label, k=k)
        rows.append({
            'query_supergroup': target_name, 'query_text': query_text, 'method': method,
            'Precision@10': metric['Precision@k'], 'Recall@10': metric['Recall@k'],
            'AP@10': metric['AP@k'], 'NDCG@10': metric['NDCG@k'],
            'relevant_test_documents': int(np.count_nonzero(candidate_groups == target_label)),
        })
    return rows

retrieval_rows = []
for group_name in target_names:
    retrieval_rows.extend(evaluate_supergroup_query(
        supergroup_queries[group_name], target_names.index(group_name), group_name,
    ))
retrieval_metrics = pd.DataFrame(retrieval_rows)
display(retrieval_metrics.round(4))
metric_columns = ['Precision@10', 'Recall@10', 'AP@10', 'NDCG@10']
mean_metrics = retrieval_metrics.groupby('method', sort=False)[metric_columns].mean().rename(columns={'AP@10': 'MAP@10'})
print('Macro-average across the six queries:')
display(mean_metrics.round(4))

In [ ]:
plot_metrics = ['Precision@10', 'MAP@10', 'NDCG@10']
retrieval_fig, retrieval_axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, metric in zip(retrieval_axes, plot_metrics):
    values = mean_metrics[metric]
    bars = ax.bar(values.index, values.to_numpy(), color=plt.cm.tab10.colors[:len(values)])
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_title(metric)
    ax.set_ylabel('Macro-average score')
    ax.set_ylim(0, min(1.15, max(0.1, float(values.max()) * 1.2)))
    ax.grid(axis='y', alpha=0.3)
retrieval_fig.suptitle('Held-out retrieval comparison by method')
retrieval_fig.tight_layout()
# display(retrieval_fig)

## 7. Check how source categories align within each group

This training-only diagnostic compares TF-IDF centroids for original categories. For each supergroup, it reports average cosine similarity among member categories and between member categories and categories assigned elsewhere. It helps assess the grouping hypothesis; it does not change group membership or model fitting.

In [ ]:
category_centroids = np.vstack([
    np.asarray(X_train_tfidf[original_y_train == category_id].mean(axis=0)).ravel()
    for category_id in range(len(source_category_names))
])
category_centroid_similarity = cosine_similarity(category_centroids)
similarity_rows = []
for group_name, members in SUPERGROUPS.items():
    member_ids = [source_category_names.index(name) for name in members]
    other_ids = [i for i, name in enumerate(source_category_names) if source_to_supergroup[name] != group_name]
    within = [category_centroid_similarity[i, j] for pos, i in enumerate(member_ids) for j in member_ids[pos + 1:]]
    between = [category_centroid_similarity[i, j] for i in member_ids for j in other_ids]
    group_mean = category_centroid_similarity[member_ids].mean(axis=0)
    closest_external = max(other_ids, key=lambda i: group_mean[i])
    similarity_rows.append({
        'supergroup': group_name, 'source_categories': len(member_ids),
        'mean_within_group_cosine': float(np.mean(within)),
        'mean_to_other_groups_cosine': float(np.mean(between)),
        'closest_external_category': source_category_names[closest_external],
    })
supergroup_similarity = pd.DataFrame(similarity_rows)
display(supergroup_similarity.round(4))

## 8. PLSA topic-mixture probe

After the retrieval comparison, we examine PLSA's document-topic probabilities directly. We create mixed documents by sampling words from `rec.sport.baseball` and `talk.politics.guns` training documents at known proportions, then fold each mixture into PLSA. The plot compares the source-word proportion with the probability of the topic most associated with baseball. These are topic probabilities, not retrieval scores or guaranteed recovery of source ratios.

In [ ]:
sports_category = 'rec.sport.baseball'
politics_category = 'talk.politics.guns'
sports_class = source_category_names.index(sports_category)
politics_class = source_category_names.index(politics_category)
source_topic_means = np.vstack([
    theta_train[original_y_train == category_id].mean(axis=0)
    for category_id in range(len(source_category_names))
])
sports_topic = int(source_topic_means[sports_class].argmax())
politics_topic = int(source_topic_means[politics_class].argmax())
print('Sports-associated topic:', sports_topic)
print('Politics-associated topic:', politics_topic)
if sports_topic == politics_topic:
    print('Warning: both categories map to the same dominant topic; interpret cautiously.')

mixture_rng = np.random.default_rng(SEED + 77)
mixture_ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
mixture_rows = []
for ratio in mixture_ratios:
    for replicate in range(20):
        mixed_counts = make_mixed_document(
            X_train, original_y_train, sports_class, politics_class,
            ratio, mixture_rng, n_tokens=300,
        )
        estimate = best_plsa.fold_in(mixed_counts, max_iter=100)[0]
        mixture_rows.append({
            'source_category_a': sports_category, 'source_category_b': politics_category,
            'actual_sports_word_ratio': ratio, 'actual_politics_word_ratio': 1 - ratio,
            'estimated_sports_topic': estimate[sports_topic],
            'estimated_politics_topic': estimate[politics_topic], 'replicate': replicate,
        })
mixture_df = pd.DataFrame(mixture_rows)
mixture_summary = mixture_df.groupby('actual_sports_word_ratio').agg(
    estimated_sports_mean=('estimated_sports_topic', 'mean'),
    estimated_sports_std=('estimated_sports_topic', 'std'),
    estimated_politics_mean=('estimated_politics_topic', 'mean'),
    estimated_politics_std=('estimated_politics_topic', 'std'),
).reset_index()
display(mixture_summary.round(4))

mixture_fig, mixture_ax = plt.subplots(figsize=(7, 5))
mixture_ax.plot([0, 1], [0, 1], 'k--', label='Ideal reference')
mixture_ax.errorbar(
    mixture_summary['actual_sports_word_ratio'],
    mixture_summary['estimated_sports_mean'],
    yerr=mixture_summary['estimated_sports_std'], marker='o', capsize=3,
    label='Estimated sports-topic probability',
)
mixture_ax.set_xlabel('Actual sports word-token proportion')
mixture_ax.set_ylabel('Estimated sports-associated topic probability')
mixture_ax.set_title('PLSA topic mixtures for synthetic documents')
mixture_ax.legend()
mixture_ax.grid(alpha=0.3)
mixture_fig.tight_layout()
# display(mixture_fig)

## 9. Save outputs

The experiment tables, figures, and configuration are saved in `results/six_category_experiment/`. Rerunning this notebook replaces the previous outputs for this experiment.

In [ ]:
retrieval_metrics.to_csv(OUTPUT_DIR / 'retrieval_metrics_by_query.csv', index=False)
mean_metrics.to_csv(OUTPUT_DIR / 'retrieval_metrics_mean.csv')
restart_summary.to_csv(OUTPUT_DIR / 'restart_summary.csv', index=False)
topic_words.to_csv(OUTPUT_DIR / 'topic_words.csv', index=False)
pd.DataFrame({
    'iteration': np.arange(1, len(best_plsa.log_likelihoods) + 1),
    'training_log_likelihood': best_plsa.log_likelihoods,
}).to_csv(OUTPUT_DIR / 'plsa_convergence.csv', index=False)
topic_alignment.to_csv(OUTPUT_DIR / 'plsa_topic_group_alignment.csv', index=False)
supergroup_similarity.to_csv(OUTPUT_DIR / 'supergroup_centroid_similarity.csv', index=False)
convergence_fig.savefig(OUTPUT_DIR / 'plsa_convergence.png', dpi=160, bbox_inches='tight')
retrieval_fig.savefig(OUTPUT_DIR / 'retrieval_metrics_comparison.png', dpi=160, bbox_inches='tight')
mixture_df.to_csv(OUTPUT_DIR / 'synthetic_mixture_replicates.csv', index=False)
mixture_summary.to_csv(OUTPUT_DIR / 'synthetic_mixture_summary.csv', index=False)
mixture_fig.savefig(OUTPUT_DIR / 'synthetic_mixture_topic_probabilities.png', dpi=160, bbox_inches='tight')

run_configuration = {
    'experiment': 'Six broad subject groups',
    'seed': SEED, 'test_size': TEST_SIZE, 'n_topics': K,
    'n_restarts': N_RESTARTS, 'max_iter': MAX_ITER, 'tolerance': TOLERANCE,
    'max_features': MAX_FEATURES, 'min_df': MIN_DF, 'max_df': MAX_DF,
    'retrieval_k': RETRIEVAL_K, 'documents': len(documents),
    'supergroups': SUPERGROUPS,
    'topic_mixture_categories': [sports_category, politics_category],
}
with (OUTPUT_DIR / 'configuration.json').open('w', encoding='utf-8') as handle:
    json.dump(run_configuration, handle, indent=2)
print(f'Saved six-category experiment outputs to {OUTPUT_DIR.resolve()}')